# Chapter 3 — Description Logics
### Notebook 5 · Agentic lab — naming the logic, and paying for soundness

*Book reference: Extends §3.2–3.3*

Two firsts for this course. The agent's oracle is **free and always right**, so failures label themselves — the cleanest self-improvement setting we have. And the MDP is **stochastic**, because the choice is between a cheap guess and an expensive certainty.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch03_toolkit as dl
import pandas as pd
A = dl.Atomic
logging.getLogger("dspy").setLevel(logging.WARNING)

In [3]:
import ch03_agentic as AG
from oe_course import evaluation as ev, llm, mdp, optimize as opt
from oe_course.skills import Skill
from oe_course.selfimprove import SelfImprovingSkill
import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


**By the end of this notebook you can:**

1. Expose a reasoner as a **function tool** and make an agent use it instead of guessing.
2. Build a dataset whose verdicts are **balanced**, and see why an unbalanced one silently teaches nothing.
3. Model the sound-but-costly / cheap-but-unsound choice as a **stochastic MDP** and solve it exactly.
4. Run a self-improvement loop where the oracle supplies the labels.

> **Prerequisite:** the Chapter 1 agentic lab. The discipline is the same; the task changes.

## 1. Tools: the reasoner is one of them

The interesting tool here is `check_subsumption`. Its description tells the agent *when* to reach for it — "instead of reasoning by eye" — because the failure mode this lab is built around is an agent that answers from the shape of the axioms rather than from a proof.

In [4]:
ctx = AG.Ch3Context()
tools = {t.name: t for t in AG.build_toolset(ctx)}
for name, t in tools.items():
    print(f'{name:22s} {list(t.args_schema.model_json_schema().get("properties", {}))}')
    print(f'{"":22s} {t.description.splitlines()[0]}')

load_kb                ['name']
                       Load a named knowledge base from the chapter's case set.
list_kbs               []
                       List the available knowledge base names.
show_axioms            []
                       Show the loaded knowledge base's axioms and role facts.
dl_expressivity        []
                       Report which DL constructors the loaded KB uses, and its DL name.
check_satisfiability   ['concept_a', 'concept_b']
                       Is `concept_a` (optionally conjoined with `concept_b`) satisfiable?
check_subsumption      ['sub', 'sup']
                       Does the KB entail that `sub` is subsumed by `sup`?
tableau_trace          ['concept_a', 'concept_b']
                       Return the tableau expansion trace -- the proof behind a verdict.


In [5]:
print(tools['load_kb'].invoke({'name': 'wildlife'}))
print(tools['dl_expressivity'].invoke({}))
print(tools['check_subsumption'].invoke({'sub': 'Giraffe', 'sup': 'Animal'}))
print(tools['check_satisfiability'].invoke({'concept_a': 'Giraffe',
                                            'concept_b': 'Carnivore'}))
print('\ntrajectory:', ctx.log.names())

{"loaded": "wildlife", "axioms": 6}
{"dl": "ALC", "constructors": ["conjunction", "existential", "negation-atomic", "qualified-existential", "universal"]}
{"subsumed": true}
{"satisfiable": false, "steps": 75, "branches": 27}

trajectory: ['load_kb', 'dl_expressivity', 'check_subsumption', 'check_satisfiability']


## 2. The dataset, and why its balance matters

Ten knowledge bases. Each asks for a DL name **and** a subsumption verdict.

The verdicts are deliberately **half true and half false**. An all-true set — which is what you get if you write the queries carelessly — would let the strategy *"assume it follows"* score full marks on that half. The rule about actually running the reasoner would never be punished, and the optimiser would never learn it. **A dataset that cannot punish a mistake cannot teach it.**

In [6]:
all_examples = AG.build_dataset('all')
print(pd.DataFrame([{'id': e.id, 'DL': e.gold_dl, 'query': e.query,
                     'holds': e.gold_subsumption} for e in all_examples]
                   ).to_string(index=False))
verdicts = [e.gold_subsumption for e in all_examples]
print(f'\nbalance: {sum(verdicts)} hold, {len(verdicts) - sum(verdicts)} do not')

               id   DL                         query  holds
   basic-taxonomy  ALC       Dog subsumed by Animal?   True
   vegetarian-alc  ALC Vegetarian subsumed by Vegan?  False
 parts-transitive    S    Twig subsumed by TreePart?   True
 family-hierarchy ALCH    Parent subsumed by Person?  False
    child-inverse ALCI     Child subsumed by Person?  False
trilogy-qualified ALCQ     Trilogy subsumed by Book?  False
 busy-unqualified ALCN  Overloaded subsumed by Busy?   True
         wildlife  ALC   Giraffe subsumed by Animal?   True
 supervision-shiq SHIQ Manager subsumed by Employee?  False
     cyclic-nodes  ALC        Head subsumed by Node?   True

balance: 5 hold, 5 do not


In [7]:
train, dev = AG.build_dataset('train'), AG.build_dataset('dev')
print('train:', [e.id for e in train])
print('dev  :', [e.id for e in dev])
print('\nThe split is stratified (alternating), not sequential: a first-six/\n'
      'last-four cut would put every transitive and inverse case in train and\n'
      'leave dev nearly trivial, so the held-out score would flatter the agent.')

train: ['basic-taxonomy', 'parts-transitive', 'child-inverse', 'busy-unqualified', 'supervision-shiq']
dev  : ['vegetarian-alc', 'family-hierarchy', 'trilogy-qualified', 'wildlife', 'cyclic-nodes']

The split is stratified (alternating), not sequential: a first-six/
last-four cut would put every transitive and inverse case in train and
leave dev nearly trivial, so the held-out score would flatter the agent.


In [8]:
print('KB as the agent sees it (no logic named for it):\n')
print(dev[1].kb)
print('\nquery:', dev[1].query)

KB as the agent sees it (no logic named for it):

axiom: Parent <= exists hasChild.Person
role hierarchy: ['hasChild sub-role of hasRelative']

query: Parent subsumed by Person?


## 3. Baseline and GEPA

The metric gives half a mark for the DL name and half for the verdict, and names the specific letter that was missed — `s-for-transitive`, `i-for-inverse` and so on — so the optimiser learns *which* rule of §3.2 it broke.

In [9]:
lm = llm.configure_dspy(AG.DL_RULEBOOK, AG.dl_responder)
baseline = AG.DLProgram()
example = train[-1]
pred = baseline(**example.inputs())
print('KB id      :', example.id)
print('answered   :', pred.dl, '/', pred.subsumption)
print('correct    :', example.gold_dl, '/', example.gold_subsumption)
report = AG.dl_scorer(example, pred)
print('score      :', report.score)
for n in report.notes:
    print('   ', n)
print('violated   :', report.violated)

KB id      : supervision-shiq
answered   : ALCN / true
correct    : SHIQ / False
score      : 0.0
    DL name wrong: answered 'ALCN', correct is 'SHIQ'.
    Subsumption verdict wrong: answered True, the reasoner says False.
    dl_ok=0 verdict_ok=0
violated   : ['s-for-transitive', 'h-for-role-hierarchy', 'i-for-inverse', 'q-for-qualified-number', 'use-the-reasoner']


In [10]:
before = ev.evaluate_dataset(baseline, dev, AG.dl_scorer)
print('BEFORE:', before['mean_score'], before['violations'])

BEFORE: 0.5 {'use-the-reasoner': 3, 'h-for-role-hierarchy': 1, 'q-for-qualified-number': 1}


In [11]:
gepa_metric = ev.make_gepa_metric(AG.dl_scorer, AG.DL_RULEBOOK)
reflect = llm.reflection_lm(AG.DL_RULEBOOK, AG.dl_responder)
tuned = opt.run_gepa(baseline, train, gepa_metric, valset=train,
                     max_metric_calls=80, reflection_lm=reflect)
result = opt.compare(AG.DLProgram(), tuned, dev, AG.dl_scorer)
print(result.report())

2026/08/16 12:05:42 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 80 metric calls of the program. This amounts to 8.00 full evals on the train+val set.


2026/08/16 12:05:42 INFO dspy.teleprompt.gepa.gepa: Using 5 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


GEPA Optimization:   0%|          | 0/80 [00:00<?, ?rollouts/s]

2026/08/16 12:05:42 INFO dspy.evaluate.evaluate: Average Metric: 2.5 / 5 (50.0%)


2026/08/16 12:05:42 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.5


...

2026/08/16 12:05:43 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 2 (100.0%)


2026/08/16 12:05:43 INFO dspy.teleprompt.gepa.gepa: Iteration 34: All subsample scores perfect. Skipping.


2026/08/16 12:05:43 INFO dspy.teleprompt.gepa.gepa: Iteration 34: Reflective mutation did not propose a new candidate


GEPA Optimization:  98%|█████████▊| 78/80 [00:01<00:00, 55.37rollouts/s]

mean score  0.500  ->  1.000   (delta +0.500)
violations  {'use-the-reasoner': 3, 'h-for-role-hierarchy': 1, 'q-for-qualified-number': 1}
        ->  {}

instruction diff:
--- instruction (before)
+++ instruction (after)
@@ -1 +1,6 @@
 You are a description logic expert. Given the knowledge base, say which description logic it needs and answer the subsumption query.
+- RULE s-for-transitive: When any role is transitive the base logic is S, not ALC: S abbreviates ALC extended with transitive roles.
+- RULE h-for-role-hierarchy: Append H when the knowledge base declares a sub-role axiom.
+- RULE i-for-inverse: Append I when any role appears inverted (written r-).
+- RULE q-for-qualified-number: Append Q for number restrictions with a filler concept (>=n r.C); use N only for unqualified ones (>=n r).
+- RULE use-the-reasoner: Never guess a subsumption. Run the reasoner: C is subsumed by D exactly when C and not D is unsatisfiable.


In [12]:
found = AG.DL_RULEBOOK.active_in(result.instruction_after)
print('rules discovered:', sorted(found))
print('rules missed    :', sorted(set(AG.DL_RULEBOOK.ids) - found))

rules discovered: ['h-for-role-hierarchy', 'i-for-inverse', 'q-for-qualified-number', 's-for-transitive', 'use-the-reasoner']
rules missed    : []


> Note what `use-the-reasoner` being discovered actually means: the optimiser learned to make the agent **call a tool** rather than answer from the prompt. That is prompt optimisation changing *behaviour*, not just wording — and it only happened because half the verdicts were false.

## 4. A stochastic MDP: when is the reasoner worth calling?

Every MDP so far has had deterministic transitions. This one does not.

The agent answers a series of subsumption queries. For each it may:

* **guess** — free, correct with probability `pᵢ` (a structural heuristic such as "it is asserted in the hierarchy, so it must follow");
* **reason** — always correct, costs `reasoner_cost`, and consumes one unit of a limited budget.

| | |
|---|---|
| **S** | which query we are on, and how much budget is spent |
| **A** | `guess` or `reason` |
| **T** | **stochastic** — guessing lands in the same next state, but the reward is Bernoulli |
| **R** | 1 for a correct answer; `1 − cost` for reasoning |

This is §3.3's complexity discussion as a decision problem: soundness is a purchase, and the question is where to spend.

In [13]:
accuracy = [0.9, 0.5, 0.95, 0.6]     # heuristic reliability, per query
M = AG.ReasoningBudgetMDP(accuracy, reasoner_cost=0.2, budget=2)
V, pi = mdp.value_iteration(M)
print(f'V*(s0) = {V[M.initial_state()]:.3f}')
print('optimal plan:', M.optimal_plan(pi))
print('heuristic accuracies:', accuracy)
print(f'\nalways guess : {sum(accuracy):.3f}')
print(f'always reason: {len(accuracy) * (1 - 0.2):.3f}  (if budget allowed)')

V*(s0) = 3.450
optimal plan: ['guess', 'reason', 'guess', 'reason']
heuristic accuracies: [0.9, 0.5, 0.95, 0.6]

always guess : 2.950
always reason: 3.200  (if budget allowed)


The optimal policy spends its two reasoner calls on queries **2 and 4** — the ones where the heuristic is least reliable (0.5 and 0.6) — and guesses on the two where it is nearly always right (0.9, 0.95). Nobody told it that rule; value iteration derived it from the reward.

The threshold is worth stating exactly: **call the reasoner when `1 − cost > pᵢ`**, i.e. when the heuristic's error rate exceeds the reasoner's price.

In [14]:
rows = []
for cost in [0.0, 0.1, 0.3, 0.5, 0.7]:
    Mc = AG.ReasoningBudgetMDP(accuracy, reasoner_cost=cost, budget=4)
    Vc, pic = mdp.value_iteration(Mc)
    plan = Mc.optimal_plan(pic)
    rows.append({'reasoner_cost': cost, 'V*': round(Vc[Mc.initial_state()], 3),
                 'reasoner calls': plan.count('reason'), 'plan': ' '.join(plan)})
print(pd.DataFrame(rows).to_string(index=False))
print('\nWith an unlimited budget the agent still declines to reason when the\n'
      'price exceeds the heuristic error rate. "Always be sound" is not the\n'
      'optimal policy under a cost model -- which is uncomfortable, and true.')

 reasoner_cost   V*  reasoner calls                        plan
           0.0 4.00               4 reason reason reason reason
           0.1 3.65               2   guess reason guess reason
           0.3 3.25               2   guess reason guess reason
           0.5 2.95               0     guess guess guess guess
           0.7 2.95               0     guess guess guess guess

With an unlimited budget the agent still declines to reason when the
price exceeds the heuristic error rate. "Always be sound" is not the
optimal policy under a cost model -- which is uncomfortable, and true.


## 5. Self-improvement, with a free oracle

This is the chapter where self-improvement is genuinely easy: the tableau settles every question, so **every failure labels itself**. No human in the loop, no annotation budget.

The promotion gate still matters. An oracle removes the labelling problem; it does not remove the risk of adopting a candidate that happens to score well on the data it was tuned on.

In [15]:
skill = Skill(
    name='dl-analyst',
    description='Name the DL a knowledge base needs and answer subsumption soundly.',
    build=lambda instruction: AG.DLProgram(instruction),
    scorer=AG.dl_scorer,
    dataset=dev,
    instruction=AG.BASELINE_INSTRUCTION,
    tools=['load_kb', 'dl_expressivity', 'check_subsumption', 'tableau_trace'],
)
skill.evaluate()
print(skill.card())

SKILL  dl-analyst  (v1)
       Name the DL a knowledge base needs and answer subsumption soundly.
tools  load_kb, dl_expressivity, check_subsumption, tableau_trace
score  0.500 on 5 examples
history:
  * v1 0.500 initial instruction


In [16]:
sis = SelfImprovingSkill(
    skill, holdout=dev,
    optimise=lambda prog, tr: opt.run_gepa(prog, tr, gepa_metric,
                                          max_metric_calls=60, reflection_lm=reflect),
    min_gain=0.01)
sis.run_all(train)
print(sis.report())

self-improvement history for skill 'dl-analyst'
  (no rounds run)
  experience: 5 episodes, 3 failures
  violations: {'s-for-transitive': 2, 'i-for-inverse': 2, 'use-the-reasoner': 2, 'h-for-role-hierarchy': 1, 'q-for-qualified-number': 1}


In [17]:
print(sis.improve())
print(sis.improve())
print()
print(skill.card())

2026/08/16 12:05:45 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 60 metric calls of the program. This amounts to 10.00 full evals on the train+val set.


2026/08/16 12:05:45 INFO dspy.teleprompt.gepa.gepa: Using 3 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


GEPA Optimization:   0%|          | 0/60 [00:00<?, ?rollouts/s]

2026/08/16 12:05:45 INFO dspy.evaluate.evaluate: Average Metric: 0.5 / 3 (16.7%)


2026/08/16 12:05:45 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.16666666666666666


...

2026/08/16 12:05:48 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 2 (100.0%)


2026/08/16 12:05:48 INFO dspy.teleprompt.gepa.gepa: Iteration 29: All subsample scores perfect. Skipping.


2026/08/16 12:05:48 INFO dspy.teleprompt.gepa.gepa: Iteration 29: Reflective mutation did not propose a new candidate


GEPA Optimization:  98%|█████████▊| 59/60 [00:01<00:00, 53.21rollouts/s]

round 2: holdout 1.000 -> 1.000 [rejected] gain +0.000 < min_gain 0.01; keeping v2

SKILL  dl-analyst  (v2)
       Name the DL a knowledge base needs and answer subsumption soundly.
tools  load_kb, dl_expressivity, check_subsumption, tableau_trace
score  1.000 on 5 examples
history:
    v1 0.500 initial instruction
  * v2 1.000 round 1: +0.500 on holdout, trained on 3 examples


> The second round is **rejected** — no gain on the holdout, so the skill stays where it is. A self-improvement loop that never rejects a candidate is not improving; it is drifting.

### Exercise 5.1 — Break the dataset on purpose

Rebuild the dataset with **all verdicts true**, re-run GEPA, and report whether `use-the-reasoner` is still discovered. Explain the result.

> **Hint.** Filter `train` down to the examples whose gold verdict is True.

In [18]:
# YOUR CODE HERE


<details>
<summary>Solution 5.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [19]:
import copy
biased_train = []
for e in train:
    if e.gold_subsumption:
        biased_train.append(e)
print(f'biased train has {len(biased_train)} examples, all with verdict True')

tuned_biased = opt.run_gepa(AG.DLProgram(), biased_train, gepa_metric,
                            valset=biased_train, max_metric_calls=60,
                            reflection_lm=reflect)
found_biased = AG.DL_RULEBOOK.active_in(opt.instruction_of(tuned_biased))
print('rules discovered:', sorted(found_biased))
print('use-the-reasoner found?', 'use-the-reasoner' in found_biased)
assert 'use-the-reasoner' not in found_biased
print('\nWith every verdict true, guessing "true" is never wrong, so the metric\n'
      'never complains and the rule is never learned. The agent would then fail\n'
      'silently in production on the first negative case. Dataset balance is not\n'
      'hygiene -- it decides what your agent is capable of learning.')

2026/08/16 12:05:48 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 60 metric calls of the program. This amounts to 10.00 full evals on the train+val set.


2026/08/16 12:05:48 INFO dspy.teleprompt.gepa.gepa: Using 3 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


biased train has 3 examples, all with verdict True


GEPA Optimization:   0%|          | 0/60 [00:00<?, ?rollouts/s]

2026/08/16 12:05:48 INFO dspy.evaluate.evaluate: Average Metric: 2.5 / 3 (83.3%)


...

2026/08/16 12:05:49 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 2 (100.0%)


2026/08/16 12:05:49 INFO dspy.teleprompt.gepa.gepa: Iteration 26: All subsample scores perfect. Skipping.


2026/08/16 12:05:49 INFO dspy.teleprompt.gepa.gepa: Iteration 26: Reflective mutation did not propose a new candidate


GEPA Optimization:  97%|█████████▋| 58/60 [00:00<00:00, 59.84rollouts/s]


rules discovered: ['s-for-transitive']
use-the-reasoner found? False

With every verdict true, guessing "true" is never wrong, so the metric
never complains and the rule is never learned. The agent would then fail
silently in production on the first negative case. Dataset balance is not
hygiene -- it decides what your agent is capable of learning.


### Exercise 5.2 — Find the price at which soundness stops paying

For a single query with heuristic accuracy `p`, derive the reasoner cost at which guessing and reasoning are equally good, and confirm it numerically.

> **Hint.** Guessing is worth p; reasoning is worth 1 − cost.

In [20]:
# YOUR CODE HERE


<details>
<summary>Solution 5.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [21]:
rows = []
for p in [0.5, 0.7, 0.9]:
    for cost in [0.05, 0.1, 0.2, 0.3, 0.5]:
        Mx = AG.ReasoningBudgetMDP([p], reasoner_cost=cost, budget=1)
        Vx, pix = mdp.value_iteration(Mx)
        rows.append({'p': p, 'cost': cost,
                     'action': Mx.optimal_plan(pix)[0],
                     'V*': round(Vx[Mx.initial_state()], 3),
                     'predicted': 'reason' if (1 - cost) > p else 'guess'})
df = pd.DataFrame(rows)
print(df.to_string(index=False))
assert (df['action'] == df['predicted']).all()
print('\nReasoning is worth it exactly when 1 - cost > p, i.e. cost < 1 - p:\n'
      'the reasoner is worth its price precisely when the heuristic error rate\n'
      'exceeds it. Value iteration and the algebra agree on every row.')

  p  cost action   V* predicted
0.5  0.05 reason 0.95    reason
0.5  0.10 reason 0.90    reason
0.5  0.20 reason 0.80    reason
0.5  0.30 reason 0.70    reason
0.5  0.50  guess 0.50     guess
0.7  0.05 reason 0.95    reason
0.7  0.10 reason 0.90    reason
0.7  0.20 reason 0.80    reason
0.7  0.30  guess 0.70     guess
0.7  0.50  guess 0.70     guess
0.9  0.05 reason 0.95    reason
0.9  0.10  guess 0.90     guess
0.9  0.20  guess 0.90     guess
0.9  0.30  guess 0.90     guess
0.9  0.50  guess 0.90     guess



Reasoning is worth it exactly when 1 - cost > p, i.e. cost < 1 - p:
the reasoner is worth its price precisely when the heuristic error rate
exceeds it. Value iteration and the algebra agree on every row.


### Exercise 5.3 — Add a rule the agent must learn from the reasoner

Add a `check-unsatisfiable-classes` rule: before answering, the agent should verify the queried concepts are satisfiable at all. Build a KB where ignoring this gives a misleading answer, and show the metric punishing it.

In [22]:
# YOUR CODE HERE


<details>
<summary>Solution 5.3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [23]:
# In the wildlife KB, Giraffe and Carnivore is unsatisfiable -- so it is
# vacuously subsumed by EVERYTHING, including nonsense.
w = dl.wildlife_tbox()
impossible = dl.And(A('Giraffe'), A('Carnivore'))
print('Giraffe-and-Carnivore satisfiable?',
      dl.satisfiable(impossible, w).satisfiable)
print('...subsumed by Plant?    ', dl.subsumes(impossible, A('Plant'), w))
print('...subsumed by Bottom?   ', dl.subsumes(impossible, dl.Bottom, w))
assert dl.subsumes(impossible, A('Plant'), w)
print('\nAn unsatisfiable concept is subsumed by everything -- ex falso quodlibet.\n'
      'So a "yes" verdict here is TRUE and completely useless: the honest answer\n'
      'is "the question is malformed, that class can have no instances". An agent\n'
      'that reports the entailment without the satisfiability check gives a\n'
      'technically correct answer that will mislead its user -- which is a good\n'
      'argument for making the check part of the skill rather than the prompt.')

Giraffe-and-Carnivore satisfiable? False
...subsumed by Plant?     True
...subsumed by Bottom?    True

An unsatisfiable concept is subsumed by everything -- ex falso quodlibet.
So a "yes" verdict here is TRUE and completely useless: the honest answer
is "the question is malformed, that class can have no instances". An agent
that reports the entailment without the satisfiability check gives a
technically correct answer that will mislead its user -- which is a good
argument for making the check part of the skill rather than the prompt.


## Chapter 3 in the course arc

| | Ch. 1 | Ch. 2 | Ch. 3 | Ch. 4 |
|---|---|---|---|---|
| task | assess | formalise | name + entail | build an axiom |
| MDP | gather evidence | search a proof | **budgeted, stochastic** | construct |
| grader | labels + judge | decision procedure | **free oracle** | labels + profiles |
| GEPA learns | reporting | quantifier semantics | DL letters + tool use | profile limits |

Chapter 3's contribution to the course argument: when a sound oracle exists, use it — for labels, for self-improvement, and as the thing your agent is *taught to call*. Chapter 4 then asks what happens when you standardise one of these logics into a language committees must agree on.